# De Politieke Mens: Nevenfuncties & Financiën

**Thema:** Extra activiteiten en inkomstenbronnen van Kamerleden.

**Kernvragen:** Hoeveel nevenfuncties hebben politici gemiddeld, in welke sectoren liggen deze, en wat zijn de gerapporteerde inkomsten per jaar en soort vergoeding?

Dit notebook verkent o.a. de silver-tabellen `persoon`, `persoon_nevenfunctie` en `persoon_nevenfunctie_inkomsten`.

In [3]:
pip install duckdb

   ---------------------------------------- 0.0/13.2 MB ? eta -:--:--
   -- ------------------------------------- 0.8/13.2 MB 4.8 MB/s eta 0:00:03
   ---- ----------------------------------- 1.6/13.2 MB 4.6 MB/s eta 0:00:03
   ------- -------------------------------- 2.6/13.2 MB 4.6 MB/s eta 0:00:03
   ----------- ---------------------------- 3.7/13.2 MB 4.5 MB/s eta 0:00:03
   --------------- ------------------------ 5.0/13.2 MB 4.9 MB/s eta 0:00:02
   ------------------- -------------------- 6.3/13.2 MB 5.1 MB/s eta 0:00:02
   ---------------------- ----------------- 7.3/13.2 MB 5.3 MB/s eta 0:00:02
   -------------------------- ------------- 8.7/13.2 MB 5.3 MB/s eta 0:00:01
   ----------------------------- ---------- 9.7/13.2 MB 5.3 MB/s eta 0:00:01
   --------------------------------- ------ 11.0/13.2 MB 5.3 MB/s eta 0:00:01
   ------------------------------------ --- 12.1/13.2 MB 5.2 MB/s eta 0:00:01
   ---------------------------------------  13.1/13.2 MB 5.3 MB/s eta 0:00:01
   

In [8]:
from pathlib import Path

import duckdb
import pandas as pd
import matplotlib.pyplot as plt

def find_database(start: Path | None = None) -> Path:
    """Find the repository DuckDB database by walking up from the current folder."""
    current = (start or Path.cwd()).resolve()
    for folder in (current, *current.parents):
        candidate = folder / "tweedekamer.duckdb"
        if candidate.is_file():
            return candidate
    raise FileNotFoundError("Could not find tweedekamer.duckdb from the current directory")


database_path = find_database()
connection = duckdb.connect(str(database_path), read_only=True)


def query_df(sql: str, parameters=None) -> pd.DataFrame:
    """Execute a SQL query and return the result as a pandas DataFrame."""
    return connection.execute(sql, parameters or []).fetchdf()


database_path

WindowsPath('C:/Users/henkl/OneDrive - NHL Stenden/Jaar 3/tweedekamer.duckdb')

## Onderwerp 1

Geef hier een beschrijving van het onderwerp of deelvraag die je wilt beantwoorden.

## Deelvraag: Wat zijn de gerapporteerde inkomsten uit nevenfunctie per politicus per jaar

**KERNVRAGEN**
- Hoeveel nevenfuncties hebben politici gemiddeld?
- Welke sectoren liggen deze nevenfuncties?
- Hoeveel ontvangt een politicus gemiddeld per jaar en in totaal aan vergoedingen uit nevenfuncties ?



## Hoeveel nevenfuncties hebben politici gemiddeld?

#### Gemiddelde , minium en maxium zonder nul waardes

Hier wordt het gemiddelde berekent zonder politici die geen nevenfuncite hebben of hebben gehad.  Verder wordt het minimum en maxium berekent. Deze antwoorden zijn over de gehele loopbaan politici.  Het is logisch dat het minium 1 is, omdat nul waardes niet worden meegenomen. 

In [323]:
gemiddelde_zonder_0 = query_df("""  
with aantal as (
select persoon_id, count(*) as aantal_per from 
silver.persoon_nevenfunctie n 
group by persoon_id
)
SELECT
   AVG(aantal_per) AS gem_nevenfunctie_p_p,MAX(aantal_per) as maximum, MIN(aantal_per) as minimum
  FROM aantal
""")
gemiddelde_zonder_0

,gem_nevenfunctie_p_p,maximum,minimum
0,4.288889,33,1


#### Gemiddelde, minium en maxium met nul waardes

Hier wordt het gemiddelde berekent alle politici. Verder wordt het minimum en maxium berekent. Deze antwoorden zijn over de gehele loopbaan politici. Het is logisch dat het minium 0 is, omdat nul waardes wel worden meegenomen.

In [325]:
gemiddelde_met_0 = query_df("""  
with aantal as (
    select p.id, count(n.persoon_id) as aantal_per 
    from silver.persoon p 
    left join silver.persoon_nevenfunctie n on n.persoon_id = p.id
    group by p.id
    )
SELECT
   AVG(aantal_per) AS gem_nevenfunctie_p_p,MAX(aantal_per) as maximum, MIN(aantal_per) as minimum
  FROM aantal
""")
gemiddelde_met_0

,gem_nevenfunctie_p_p,maximum,minimum
0,0.626014,33,0


## Wie heeft het maxium aantal nevenfuncties?

De persoon die de meeste nevenfuncties heeft gehad is Alexander Pechtold. In totaal 33. Dit betekent niet meteen dat hij er veel mee verdiend per jaar. Omdat het over de gehele loopbaan gaat. Verder zegt dit ook niet of hij per neuvenfunctie geld verdiend

In [132]:
maxium_0 = query_df("""  
with aantal as (
    select persoon_id, count(*) as aantal_neuven_functie 
    from silver.persoon_nevenfunctie n 
    group by persoon_id
)
Select voornamen,achternaam,aantal_per
From aantal a
left join silver.persoon p on a.persoon_id = p.id
where aantal_per = (SELECT MAX(aantal_per) FROM aantal)
group by aantal_per, persoon_id, voornamen,achternaam
""")
maxium_0

,voornamen,achternaam,aantal_per
0,Alexander,Pechtold,33


## sectoren op basis van omschrijving tellen

Als er na elke omschrijving wordt gekeken is niet erg nuttig, omdat er 1814 verschillende omschrijvingen zijn

In [365]:
sector_omschrijving = query_df("""select omschrijving, count(*) as aantal
from silver.persoon_nevenfunctie
group by omschrijving
order by aantal desc
""")


aantal_omschrijvingen= query_df("""select  count(distinct omschrijving) as aantal_verschillende
from silver.persoon_nevenfunctie
""")

print(aantal_omschrijvingen)
sector_omschrijving

   aantal_verschillende
0                  1814


,omschrijving,aantal
0,Lid stichtingsbestuur ondersteuning Tweede Kam...,13
1,Geen,8
2,Lid fractiebestuur D66,6
3,Voorzitter stichting ondersteuning Tweede Kame...,5
4,Lid gemeenteraad Amsterdam,4
...,...,...
1809,Reservist bij Defensie,1
1810,Ontvanger Aletta Jacobsprijs,1
1811,Lid Bestuur Nederlandse Vereniging voor de Arb...,1
1812,Lid van Vereniging Fijn Zonneplein,1


## Sectoren selecter op basis van omschrijving. 

Zelf sectoren maken, omdat er geen duidelijke sectoren in het bestand staan. Di werkt ook niet helemaal goed, omdat er 1159 niet onder deze catogoriën geplaats kan worden. Verder kan je niet met zekerheid zeggen dat het in de goede sector beland is. 


In [357]:
trefwoorden = query_df("""
with sectoren as (
select
    id, omschrijving,
    case
        when lower(omschrijving) like '%ziekenhuis%'
          or lower(omschrijving) like '%zorg%'
          or lower(omschrijving) like '%gezondheid%' then 'Zorg'
        when lower(omschrijving) like '%onderwijs%'
          or lower(omschrijving) like '%school%'
          or lower(omschrijving) like '%universiteit%'
          or lower(omschrijving) like '%hogeschool%' then 'Onderwijs'
        when lower(omschrijving) like '%bank%'
          or lower(omschrijving) like '%verzekering%'
          or lower(omschrijving) like '%financ%' then 'Financieel'
        when lower(omschrijving) like '%gemeente%'
          or lower(omschrijving) like '%provincie%'
          or lower(omschrijving) like '%waterschap%'
          or lower(omschrijving) like '%kamer%'
          or lower(omschrijving) like '%overheid%' then 'Overheid'
        when lower(omschrijving) like '%stichting%'
          or lower(omschrijving) like '%vereniging%'
          or lower(omschrijving) like '%fonds%' then 'Maatschappelijk'
        else 'Overig'
    end as sector
from silver.persoon_nevenfunctie)

select sector ,count(*)
from sectoren
group by sector
""")

trefwoorden


,sector,count_star()
0,Onderwijs,67
1,Maatschappelijk,509
2,Zorg,35
3,Overheid,138
4,Overig,1159
5,Financieel,22


## Hoevaak worden politici uitbetaald van nevenfunctie? 

In [175]:
frequentie_aantal = query_df("""
select frequentie,count(*)
from silver.persoon_nevenfunctie_inkomsten
group by frequentie""")
frequentie_aantal

,frequentie,count_star()
0,Per jaar,127
1,Per maand,153
2,Vrije invoer,8
3,None,273
4,Eenmalig,2
5,Per dag,5


## Soorten vergoedingen 

In [371]:
vergoeding = query_df("""
select 
    vergoeding_soort,
    count(*) as aantal
from silver.persoon_nevenfunctie
group by vergoeding_soort
order by aantal desc
""")
vergoeding

,vergoeding_soort,aantal
0,Onbezoldigd,1213
1,None,396
2,Bezoldigd,300
3,Onkostenvergoeding,21


**Betekenis soorten vergoedingen**
* Bezoldigd betekent dat je betaald wordt voor je werk
* Onbezoldigd betekent dat je werkt zonder loon of vergoeding
* Onkostenvergoeding belastingvrije vergoeding die een werkgever aan een werknemer of organisatie aan een vrijwilliger geeft om gemaakte zakelijke kosten te dekken.


## Hoeveel ontvangt een politicus gemiddeld per jaar en in totaal aan vergoedingen uit nevenfuncties die per jaar worden uitbetaald?

In [425]:
vergoeding_perjaar = query_df("""
with sectoren as (
    select 
        sum(i.bedrag) as som,voornamen,achternaam,p.id,i.jaar
    from silver.persoon_nevenfunctie_inkomsten i 
    inner join silver.persoon_nevenfunctie n on n.id = i.persoon_nevenfunctie_id
    inner join silver.persoon p on p.id = n.persoon_id
    where frequentie = 'Per jaar'
    group by voornamen,achternaam,i.jaar,p.id
    order by i.jaar, sum(bedrag)
    )
select voornamen,achternaam,avg(som) as gemiddelde_bedrag_per_jaar,count(som) as aantal_jaren_nevenfuncite,sum(som)
from sectoren
group by voornamen,achternaam
order by gemiddelde_bedrag_per_jaar desc
""")
vergoeding_perjaar

,voornamen,achternaam,gemiddelde_bedrag_per_jaar,aantal_jaren_nevenfuncite,sum(som)
0,Pieter Herman,Omtzigt,78587.473333,3,235762.42
1,Dennis,Ram,71037.000000,1,71037.00
2,Johannes Jacobus Theodorus Maria,Valize,45622.000000,1,45622.00
3,Klaas Henricus Dominicus Maria,Dijkhoff,17909.030000,4,71636.12
4,Ton Maarten Christofoor,Elias,12000.000000,18,216000.00
5,Eline Cornelia,Vedder,11681.000000,1,11681.00
6,Maurits Rolof Hubertus Marie,Martels,10000.000000,3,30000.00
7,Romke Harke,Jong,8100.000000,2,16200.00
8,Johannes Jacobus,Klink,6946.666667,3,20840.00
9,Jacob Lubertus,Geurts,6521.000000,1,6521.00


## Hoeveel ontvangt een politicus gemiddeld per jaar en in totaal aan vergoedingen uit nevenfuncties? 

In [429]:
vergoeding_perjaar = query_df("""
with vergoeding as (select 
    sum(bedrag) as totaal_bedrag ,jaar,voornamen,achternaam
from silver.persoon_nevenfunctie_inkomsten i 
inner join silver.persoon_nevenfunctie n on n.id = i.persoon_nevenfunctie_id
inner join silver.persoon p on p.id = n.persoon_id
where i.bedrag is not null
group by voornamen,achternaam,jaar
order by sum(bedrag) desc)
select voornamen,achternaam,sum(totaal_bedrag) as totaal ,avg(totaal_bedrag) as gemiddelde ,count(jaar) as aantal_jaar
from vergoeding
group by voornamen,achternaam
order by sum(totaal_bedrag) desc
""")
vergoeding_perjaar

,voornamen,achternaam,totaal,gemiddelde,aantal_jaar
0,Ton Maarten Christofoor,Elias,298796.00,16599.777778,18
1,Pieter Herman,Omtzigt,235762.42,78587.473333,3
2,Andreas Johannes,Flach,86799.00,21699.750000,4
3,Dennis,Ram,71737.00,35868.500000,2
4,Klaas Henricus Dominicus Maria,Dijkhoff,71636.12,17909.030000,4
...,...,...,...,...,...
96,Wytske Liselotte,Postma,475.00,475.000000,1
97,Sandra Theodora Petronella Hendrika,Palmen,407.79,407.790000,1
98,Peter Johannes,Valstar,399.93,399.930000,1
99,Alexander,Pechtold,80.00,80.000000,1


## Onderwerp 2

...